# Which dataset characteristics actually matter

Stage 2f. Goodness-of-fit was shown against every statistical characteristic as
a rolling average over 250 datasets each side, in a 21-panel figure per arm.
That presentation carries no uncertainty band, shows no data density, and gives
21 MARGINAL views of characteristics that are correlated with each other, so it
overstates how many independent effects there are.

**This notebook is separate from the other three on purpose.** It reads the
characteristics from notebook 1, the goodness-of-fit and cross-validated scores
from notebook 2, and the run against the true parent distributions from
notebook 3, and it computes nothing that needs the probabilistic LCA machinery.
It runs in about fifteen minutes, which is what makes the reduction something
that can be iterated on.

**THERE ARE THREE TARGETS AND THEY ANSWER DIFFERENT QUESTIONS.**

| target | the question | where the answer lives |
|---|---|---|
| the LEVEL of a method's own score | how far is this fitted curve from its target | dominated by dispersion and dataset size, because every method gets worse on spread, small data |
| the CHOICE between two families | which method should I use | whose shape assumption fits: skewness, kurtosis, lognormality, outlier weight |
| the ANSWER a pLCA gives | how wrong is the final number | between the two, and closest to size |

Reporting the LEVEL ranking as an answer to the CHOICE question is a mistake
this analysis made in its first pass, and the two are now separated.

In [ ]:
import json
import os
import sys
import time
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import product

sys.path.insert(0, '../src')

# Drop this project's own modules before importing them, so a kernel that has
# been open across an edit picks up the edit.
for _name in [_n for _n in list(sys.modules)
              if _n in ('comparison', 'corpus', 'coverage', 'customstats',
                        'empirical', 'families', 'figstyle', 'fitting',
                        'genconfig', 'metricreduction', 'modality')]:
    del sys.modules[_name]

import comparison
import corpus
import coverage as coverage_mod
import figstyle
import metricreduction as metricred

# Screen resolution for the figure, print resolution for the file.
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['savefig.dpi'] = 300

OUT = '../outputs'

# All randomness in this notebook comes from this Generator.
SEED = 42
rng = np.random.default_rng(SEED)

PE = ['Normal', 'Lognormal', 'KDE']
WT = ['Uniform', 'Variable']
PEWT = [f'{pe}, {wt}' for pe, wt in product(PE, WT)]
dct_colors = {pewt: c for pewt, c
              in zip(PEWT, sns.color_palette('Paired', len(PEWT)))}
with open('../src/dct_metriclabels.json') as f:
    dct_metriclabels = json.load(f)
print(f'{len(PEWT)} methods, seed {SEED}')

In [ ]:
# THE THREE INPUTS, each written by an earlier notebook, and the candidate set.
#
# 25 candidates: the 21 characteristics the old figure drew, the two
# visible-mode counts, and the author's modality index at the bandwidth the
# study FITS -- which is the only version of that measure never tested, the
# existing column being the same index at Scott's rule.
_emp = pd.read_excel(f'{OUT}/tables/TABLE_EmpiricalECCMetrics.xlsx',
                     index_col=0).reset_index()
_emp.columns = ['dataset'] + list(_emp.columns[1:])
_emp['arm'] = 'empirical'

_metrics_all, _, _runmeta = corpus.load_corpus(with_values=False)
_syn = _metrics_all[~_metrics_all.is_probe].copy()
_syn['arm'] = 'synthetic'
_shared = [c for c in _emp.columns if c in _syn.columns]
red_chars = pd.concat([_emp[_shared], _syn[_shared]], ignore_index=True)

red_scores = pd.read_csv(f'{OUT}/tables/TABLE_TargetComparison.csv')
red_modes = pd.read_csv(f'{OUT}/tables/TABLE_VisibleModes.csv')
red_truth = pd.read_csv(f'{OUT}/tables/TABLE_PLCATruth.csv.gz')

red_frame, red_metrics = metricred.assemble(red_chars, red_scores,
                                            truth=red_truth, modes=red_modes)
red_metrics = list(red_metrics) + [
    m for m in metricred.MODE_METRICS + metricred.EXTRA_METRICS
    if m in red_frame.columns]

print(f"corpus {_runmeta['label']}  "
      f"{len(red_frame):,} rows over {red_frame.dataset.nunique():,} datasets")
print(f'{len(red_metrics)} candidate characteristics')
for _t in ('w1', 'w1_cv', 'w1_market', 'err_eci_mean', 'err_eci_rank_1',
           'err_ui', 'err_eci_p95'):
    if _t in red_frame:
        _c = red_frame.groupby('arm')[_t].apply(lambda s: int(s.notna().sum()))
        print(f'  {_t:16s} ' + '  '.join(f'{k} {v:,}' for k, v in _c.items()))

## What the models are allowed to forget

In [ ]:
# WHAT A COMPLETE-CASE MODEL WOULD THROW AWAY, before any model is fitted.
#
# Two different things remove the smallest datasets and neither announces
# itself. Excess kurtosis divides by (n-1)(n-2)(n-3) so it is undefined below
# four values; a visible mode count needs eight. Both models below keep every
# row -- the additive one imputes with a missingness indicator, the boosted one
# splits on missingness natively -- and these tables are what say why that
# mattered.
red_missing = metricred.missingness_by_band(red_frame, red_metrics)
red_missing.to_csv(f'{OUT}/tables/TABLE_ReductionMissingness.csv', index=False)
red_cost = metricred.complete_case_cost(red_frame, red_metrics)
red_cost.to_csv(f'{OUT}/tables/TABLE_ReductionCompleteCaseCost.csv',
                index=False)
display(red_cost)

# The SECOND exclusion, which the table above cannot see: the cross-validated
# empirical target is undefined below n = 10, because half of a nine-value
# dataset is four values. So the out-of-sample reduction on the real arm has
# nothing to say about the smallest band, and those claims rest on the
# in-sample target instead.
red_used = metricred.rows_used_by_band(
    red_frame, ('w1', 'w1_cv', 'w1_market', 'err_eci_mean', 'err_eci_rank_1',
                'err_ui', 'err_eci_p95'))
red_used.to_csv(f'{OUT}/tables/TABLE_ReductionRowsUsed.csv', index=False)
display(red_used[red_used.method == 'KDE, Uniform']
        [['arm', 'target', 'rows_total', 'rows_used']
         + [c for c in red_used.columns if c.startswith('used__')]])

## How many independent things are there, and what are they

In [ ]:
# THE CORRELATION STRUCTURE, and then the DIRECTIONS.
#
# The effective dimension is the participation ratio of the correlation
# eigenvalues: it would be 25 if the candidates were independent and 1 if they
# were all the same quantity. It says HOW MANY independent quantities there
# are and not WHAT they are, which is what the components below are for --
# "about four or five" is not something a reader can act on.
red_redundancy = metricred.redundancy_table(red_frame, red_metrics)
red_redundancy.to_csv(f'{OUT}/tables/TABLE_ReductionRedundancy.csv',
                      index=False)
print('EFFECTIVE DIMENSION of the candidate set')
print(red_redundancy.groupby('arm')[['effective_dimension', 'n_metrics']]
      .first().to_string())
print()
print('the most redundant pairs -- a pair above 0.9 is one quantity')
display(red_redundancy.head(10)[['arm', 'metric_a', 'metric_b', 'correlation']])

red_pca = metricred.principal_components(red_frame, red_metrics)
red_pca.to_csv(f'{OUT}/tables/TABLE_ReductionComponents.csv', index=False)
for _arm in ('empirical', 'synthetic'):
    _g = red_pca[red_pca.arm == _arm]
    if not len(_g):
        continue
    print(f'\n{_arm}: {_g.n_datasets.iloc[0]:,} complete datasets, '
          f'{_g.n_characteristics.iloc[0]} characteristics, '
          f'participation ratio {_g.participation_ratio.iloc[0]:.2f}')
    for _, r in _g.iterrows():
        print(f'  {r.component}  {r.variance_share*100:4.1f} pct '
              f'(cum {r.cumulative_share*100:4.1f})  {r.loadings}')

## The question the paper asks: which method should I use

The reduction's first pass ranked characteristics by how well they predict the
LEVEL of a single method's score. The level is dominated by dispersion and
dataset size, because every one of the six methods gets worse on spread data
and on small samples. **That is not the question the paper asks.** The paper
asks which method to use, which is the DIFFERENCE between two of them, and a
difference is about whose shape assumption fits.

Two properties of the ratio matter. Taken within a weighting scheme it CANCELS
the part of the score no estimator can remove, which makes the
uniform-to-variable distance a legitimate predictor here where on the level it
was an identity. And it is scale free, so it does not inherit the level's
dependence on how spread the data happen to be.

In [ ]:
# WHAT EACH CHARACTERISTIC ADDS TO THE CHOICE, over two bases.
#
# Over dataset size alone, which is the mechanism every earlier stage found;
# and over size AND dispersion together, which is what the level-target
# reduction said was all that mattered. A characteristic that still adds
# something over the second base is one the level target was hiding.
#
# THE BONFERRONI THRESHOLD TRAVELS WITH THE P-VALUE, because about fifteen
# candidates are tested on 127 real categories and fifteen tests at the five
# percent level expect about one false positive.
red_choice = {}
for _value, _arm in (('w1_cv', 'empirical'), ('w1_market', 'synthetic')):
    red_choice[_arm] = metricred.choice_frame(red_frame, _value)

_inc = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        for _base in (('n',), ('n', 'coeffvar')):
            _d = metricred.choice_increments(_ch, red_metrics, _arm, _wt,
                                             base=_base)
            if len(_d):
                _inc.append(_d)
red_choice_increments = pd.concat(_inc, ignore_index=True)
red_choice_increments.to_csv(
    f'{OUT}/tables/TABLE_ReductionChoiceIncrements.csv', index=False)

for _arm in ('empirical', 'synthetic'):
    for _wt in ('Uniform', 'Variable'):
        _g = red_choice_increments[
            (red_choice_increments.arm == _arm)
            & (red_choice_increments.weighting == _wt)
            & (red_choice_increments.base == 'n+coeffvar')]
        if not len(_g):
            continue
        print(f'{_arm} / {_wt}: over size AND dispersion, base R2 '
              f'{_g.r2_base.iloc[0]:.3f}, {_g.n_rows.iloc[0]} datasets')
        display(_g.head(7)[['metric', 'incremental_r2', 'p_value',
                            'survives_bonferroni']])

In [ ]:
# THE SAME QUESTION BY PERMUTATION IMPORTANCE, on both model families, so the
# ranking does not rest on one modeling assumption.
_ci = []
for _arm, _ch in red_choice.items():
    for _wt in ('Uniform', 'Variable'):
        _d = metricred.choice_importance(_ch, red_metrics, arm=_arm,
                                         weighting=_wt, rng=rng)
        if len(_d):
            _ci.append(_d)
red_choice_importance = pd.concat(_ci, ignore_index=True)
red_choice_importance.to_csv(
    f'{OUT}/tables/TABLE_ReductionChoiceImportance.csv', index=False)
print('out-of-sample R2 of the choice models themselves; an importance inside')
print('a model that predicts nothing means nothing')
display(red_choice_importance.groupby(['arm', 'weighting', 'model'])
        .model_r2.first().unstack().round(3))
print('top predictors of WHICH METHOD WINS, averaged over the models above')
display(red_choice_importance.groupby('metric')
        .agg(mean_importance=('importance', 'mean'),
             max_importance=('importance', 'max'))
        .sort_values('mean_importance', ascending=False).head(10).round(4))

## The level and the answer, for comparison

Kept because the contrast between the three targets is the finding. The level
is what the figure this notebook replaces was drawn against.

In [ ]:
# THE LEVEL AND THE ANSWER TARGETS. This is the expensive cell, about ten
# minutes: roughly fifty (arm, method, target) cells, two models each, each
# ranked by permutation importance on the held-out fold of a five-fold split.
_t0 = time.time()
FIT_TARGETS = ('w1', 'w1_cv', 'w1_market')
ANSWER_TARGETS = ('err_eci_mean', 'err_eci_rank_1', 'err_ui', 'err_eci_p95')

_plan = []
for _t in FIT_TARGETS:
    for _arm in ('empirical', 'synthetic'):
        _plan += [(_arm, _m, _t) for _m in PEWT]
for _t in ANSWER_TARGETS:
    _plan += [('synthetic', _m, _t) for _m in PEWT]

_imps = []
for _arm, _method, _target in _plan:
    if _target not in red_frame.columns:
        continue
    _sub = red_frame[(red_frame.arm == _arm) & (red_frame.method == _method)]
    if _sub[_target].notna().sum() < 60:
        continue
    _imps.append(metricred.importance(red_frame, red_metrics, _target,
                                      arm=_arm, method=_method, rng=rng))
red_importance = pd.concat([d for d in _imps if len(d)], ignore_index=True)
red_importance['target_family'] = red_importance.target.map(
    lambda t: metricred.TARGETS[t]['family'])
red_importance.to_csv(f'{OUT}/tables/TABLE_ReductionImportance.csv',
                      index=False)
print(f'{len(red_importance):,} importance rows in {time.time()-_t0:.0f}s')
display(red_importance.groupby(['target', 'arm', 'model'])
        .model_r2.first().unstack().round(3))

In [ ]:
# IS A CANDIDATE PREDICTING A TARGET, OR IS IT PART OF ONE?
#
# A fit score splits into the part an estimator can remove and a definitional
# part it cannot, and for an equal-weighted method that definitional part IS
# the uniform-to-variable distance, because every model is scored against the
# market-weighted target. Ranking it first on a LEVEL target rediscovers an
# identity. On the CHOICE target, taken within a weighting scheme, the
# identity cancels from both terms and it becomes a real predictor.
red_definitional = metricred.definitional_check(red_frame, red_scores,
                                                metrics=red_metrics)
red_definitional.to_csv(f'{OUT}/tables/TABLE_ReductionDefinitional.csv',
                        index=False)
_ident = red_definitional[red_definitional.is_an_identity]
display(_ident[['arm', 'method', 'metric', 'spearman_definitional',
                'spearman_w1', 'spearman_err_eci_mean']].round(4))

In [ ]:
# IS A CHARACTERISTIC JUST DATASET SIZE WEARING ANOTHER NAME?
red_size = metricred.size_confounding(red_frame, red_metrics)
red_size.to_csv(f'{OUT}/tables/TABLE_ReductionSizeConfounding.csv', index=False)
for _arm in ('empirical', 'synthetic'):
    print(f'{_arm}: most explained by log(n) alone')
    display(red_size[red_size.arm == _arm].head(5)
            [['metric', 'n_used', 'r2_on_log_n', 'spearman_with_log_n']])

In [ ]:
# THE SURVIVORS, by target family, with and without the definitional
# candidate. A metric's score is its mean rank across every model that
# predicted anything at all, and a model whose out-of-sample R2 is below 0.02
# contributes nothing because inside it the ordering is noise.
_all_imp = pd.concat([red_importance, red_choice_importance], ignore_index=True)
_surv = []
for _fam in ('fit', 'answer', 'choice'):
    for _trim, _tag in ((False, ''), (True, ', definitional removed')):
        _sub = _all_imp[_all_imp.target_family == _fam]
        if _trim:
            _sub = _sub[~_sub.metric.isin(metricred.DEFINITIONAL_CANDIDATES)]
        _s = metricred.rank_survivors(_sub)
        if len(_s):
            _s.insert(0, 'target_family', _fam + _tag)
            _surv.append(_s)
red_survivors = pd.concat(_surv, ignore_index=True)
red_survivors.to_csv(f'{OUT}/tables/TABLE_ReductionSurvivors.csv', index=False)
for _fam in red_survivors.target_family.unique():
    _s = red_survivors[red_survivors.target_family == _fam]
    print(_fam.upper())
    display(_s.head(7)[['metric', 'mean_rank', 'mean_importance',
                        'share_top5', 'n_models', 'survivor']])

SURVIVORS_CHOICE = list(
    red_survivors[red_survivors.target_family == 'choice, definitional removed']
    .query('survivor').metric)
print('choice survivors carried into the figures:', ', '.join(SURVIVORS_CHOICE))

In [ ]:
# WHICH CHARACTERISTICS SURVIVE ONE TARGET AND NOT ANOTHER. These are the
# interesting ones, and the level-versus-choice column is the one that
# corrects the reduction's first pass.
_ranks = (red_survivors[red_survivors.target_family.str.endswith(
              'definitional removed')]
          .assign(fam=lambda d: d.target_family.str.split(',').str[0])
          .pivot_table(index='metric', columns='fam', values='mean_rank'))
_ranks['choice_minus_fit'] = _ranks.get('choice') - _ranks.get('fit')
_ranks['answer_minus_fit'] = _ranks.get('answer') - _ranks.get('fit')
red_target_shift = _ranks.sort_values('choice_minus_fit').reset_index()
red_target_shift.to_csv(f'{OUT}/tables/TABLE_ReductionTargetShift.csv',
                        index=False)
print('negative choice_minus_fit = matters MORE for the CHOICE than for the fit')
display(red_target_shift.round(2))

## Post-stratification, modality, and coverage

In [ ]:
# POST-STRATIFIED: the same reduction on a corpus resampled to the EMPIRICAL
# size mix, because the corpus allocates datasets equally across four size
# bands for precision while the real categories are nothing like that.
_shares = metricred.empirical_size_shares(red_frame)
print('empirical size mix: ' + ', '.join(f'{k} {v:.3f}'
                                         for k, v in _shares.items()))
red_ps_frame = metricred.size_mix_resample(red_frame, _shares, rng=rng)
print(f'{red_ps_frame.dataset.nunique():,} synthetic datasets at that mix')
_ps = []
for _target in ('w1_market', 'err_eci_mean'):
    for _method in PEWT:
        _d = metricred.importance(red_ps_frame, red_metrics, _target,
                                  arm='synthetic', method=_method, rng=rng)
        if len(_d):
            _ps.append(_d)
red_importance_ps = pd.concat(_ps, ignore_index=True)
red_importance_ps['allocation'] = 'empirical size mix'
_eq = red_importance[(red_importance.arm == 'synthetic')
                     & red_importance.target.isin(('w1_market',
                                                   'err_eci_mean'))].copy()
_eq['allocation'] = 'equal'
red_poststrat = pd.concat([_eq, red_importance_ps], ignore_index=True)
red_poststrat.to_csv(f'{OUT}/tables/TABLE_ReductionPostStratified.csv',
                     index=False)
display(red_poststrat.groupby(['allocation', 'metric']).importance.mean()
        .unstack(0).sort_values('equal', ascending=False).head(10).round(4))

In [ ]:
# THE MODALITY MEASURES, HEAD TO HEAD, and there are now four of them.
#
# `modality_index` is the author's index off a SCOTT bandwidth, which is what
# it was written with and what the study stopped using in Stage 2b.
# `modality_index_fitted` is the same index at the bandwidth the study FITS,
# which is the only version never tested before Stage 2f, and the difference
# is not cosmetic: at Scott's bandwidth the index spans 1.000 to 1.074 across
# the real categories, which is what Stage 2a read as "constant at 1" when it
# set the measure aside, and at the fitted bandwidth it spans 1.000 to 1.249.
# `crit_bw_1` is Silverman's critical bandwidth and `modes_fitted` is a count.
red_mod_agree = metricred.modality_agreement(red_frame)
red_mod_agree.to_csv(f'{OUT}/tables/TABLE_ReductionModalityAgreement.csv',
                     index=False)
display(red_mod_agree[red_mod_agree.measure_b != 'share_unimodal'].round(3))

_mh = []
for _arm, _targets in (('empirical', ('w1', 'w1_cv')),
                       ('synthetic', ('w1', 'w1_market', 'err_eci_mean'))):
    _d = metricred.modality_head_to_head(red_frame, _targets, PEWT, _arm,
                                         rng=rng)
    if len(_d):
        _mh.append(_d)
red_modality = pd.concat(_mh, ignore_index=True)
red_modality.to_csv(f'{OUT}/tables/TABLE_ReductionModality.csv', index=False)
print('what each modality measure adds over a spline in log(n) ALONE')
display(red_modality.round(4))

In [ ]:
# DOES THE CORPUS COVER THE CHARACTERISTICS THAT TURN OUT TO MATTER?
#
# The generator's tuning objective matches the SHAPE of the synthetic
# characteristic distribution to the empirical one, while the study also needs
# to SPAN that space with margin so its conclusions generalize past the
# categories EC3 happens to hold. If the characteristics that predict are the
# ones the corpus reaches least far on, the generalization claim is narrower
# than the coverage figure suggests.
_cov = coverage_mod.coverage_table(
    red_chars[red_chars.arm == 'empirical'],
    red_chars[red_chars.arm == 'synthetic'],
    metrics=tuple(m for m in red_metrics if m in red_chars.columns))
_pooled = metricred.rank_survivors(
    _all_imp[~_all_imp.metric.isin(metricred.DEFINITIONAL_CANDIDATES)])
red_cov_imp = metricred.coverage_versus_importance(_cov, _pooled)
red_cov_imp.to_csv(f'{OUT}/tables/TABLE_ReductionCoverageVsImportance.csv',
                   index=False)
display(red_cov_imp.round(4))
for _k, _v in red_cov_imp.attrs.items():
    print(f'{_k}: {_v:+.3f}')

# AND IN REAL UNITS, because "margin of -0.63" is not actionable and "the
# corpus reaches 2.6 where the data reach 6.9" is.
_short = _cov[_cov.margin_above < 0].copy()
_short['short_by'] = _short.empirical_max / _short.synthetic_max
red_shortfall = _short[['metric', 'empirical_max', 'synthetic_max',
                        'short_by', 'empirical_covered']]
red_shortfall.to_csv(f'{OUT}/tables/TABLE_ReductionCoverageShortfall.csv',
                     index=False)
print('\nWHERE THE CORPUS DOES NOT REACH THE REAL MAXIMUM')
display(red_shortfall.round(3))

## The figures

In [ ]:
# THE CURVES THAT REPLACE THE ROLLING AVERAGES, as a table on disk.
#
# Equal-count bins with a within-bin percentile bootstrap carry the
# uncertainty and the counts; a LOWESS smooth bounded to the span the bins
# cover is the read through them.
_curves = []
for _arm, _values in (('empirical', ('w1', 'w1_cv')),
                      ('synthetic', ('w1', 'w1_market', 'err_eci_mean'))):
    for _value in _values:
        if _value not in red_frame.columns:
            continue
        if red_frame.loc[red_frame.arm == _arm, _value].notna().sum() < 60:
            continue
        _c = metricred.curves_for(red_frame, SURVIVORS_CHOICE, list(PEWT),
                                  value=_value, arm=_arm, rng=rng)
        if len(_c):
            _c['value'] = _value
            _curves.append(_c)
red_curves = pd.concat(_curves, ignore_index=True)
red_curves.to_csv(f'{OUT}/tables/TABLE_ReductionCurves.csv.gz', index=False,
                  compression='gzip')
print(f'{len(red_curves):,} curve rows over '
      f'{red_curves.characteristic.nunique()} characteristics')

In [ ]:
# FIGURE A: what changes which method fits better, and what does not.
#
# THE MESSAGE: of the 23 characteristics this study measures, eight change
# which method fits better under equal weights and three under market-share
# weights, once dataset size and dispersion are already accounted for. Every
# candidate is named, so nothing looks quietly dropped, and the significance
# threshold is shown rather than a ranking being offered on trust.
#
# THE TWO WEIGHTINGS ARE DRAWN SEPARATELY AND NOT AVERAGED. Averaging them and
# taking the worse p-value leaves exactly one characteristic significant, which
# hides the real pattern: the lognormal Shapiro statistic is the only one that
# matters under BOTH weightings, and the rest matter under one. That asymmetry
# is a finding, not noise to be smoothed over.
figstyle.apply()
_ci = pd.read_csv(f'{OUT}/tables/TABLE_ReductionChoiceIncrements.csv')
_g = _ci[(_ci.base == 'n+coeffvar') & (_ci.arm == 'empirical')]
_thresh = float(_g.bonferroni_threshold.iloc[0])
_order = (_g.groupby('metric').incremental_r2.max()
          .sort_values(ascending=True).index.tolist())
_W = {'Uniform': (figstyle.MUTED, 'o', 'equal weights'),
      'Variable': (figstyle.ACCENT, 'D', 'market-share weights')}

fig, ax = plt.subplots(figsize=(figstyle.FULL_WIDTH * 0.78, 4.6))
_y = np.arange(len(_order))
for _i, _m in enumerate(_order):
    _h = _g[_g.metric == _m]
    ax.plot([0, _h.incremental_r2.max()], [_i, _i], color=figstyle.FAINT,
            linewidth=0.7, zorder=0)
_counts = []
for _wt, (_col, _mk, _lab) in _W.items():
    _h = _g[_g.weighting == _wt].set_index('metric').reindex(_order)
    _v = _h.incremental_r2.to_numpy(float)
    _sig = _h.survives_bonferroni.fillna(False).to_numpy(bool)
    ax.scatter(_v[_sig], _y[_sig], s=30, marker=_mk, color=_col, zorder=3,
               linewidths=0)
    ax.scatter(_v[~_sig], _y[~_sig], s=30, marker=_mk, facecolors='none',
               edgecolors=_col, linewidths=0.9, zorder=3)
    # THE COUNT GOES IN THE EMPTY CORNER, not at the end of the series. Placed
    # at the series end it landed on top of a data point, which the style guide
    # forbids and which `figstyle.check_overlaps` now detects.
    _counts.append((f'{_lab}: {int(_sig.sum())} of {len(_order)} significant',
                    _col, _mk))
ax.set_yticks(_y)
ax.set_yticklabels([dct_metriclabels.get(m, m).replace('\n', ' ')
                    for m in _order], fontsize=6.5)
ax.set_xlabel('added explanatory power for WHICH METHOD FITS BETTER,\n'
              'once dataset size and dispersion are already in the model',
              fontsize=7.5)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=6.5)
ax.set_xlim(left=0)
for _j, (_txt, _col, _mk) in enumerate(_counts):
    ax.scatter([0.60], [0.175 - 0.045 * _j], s=30, marker=_mk, color=_col,
               linewidths=0, transform=ax.transAxes, clip_on=False)
    ax.text(0.63, 0.175 - 0.045 * _j, _txt, transform=ax.transAxes,
            ha='left', va='center', fontsize=6.5, color=_col)
ax.text(0.99, 0.03, 'filled = significant after correcting for the 23\n'
                    'characteristics tested (p < %.4f)' % _thresh,
        transform=ax.transAxes, ha='right', va='bottom', fontsize=6,
        color=figstyle.MUTED)
ax.set_title('Ten of twenty-three characteristics change which method fits '
             'better; only one does so\nunder both weighting schemes',
             fontsize=9.5, loc='left')
plt.savefig(f'{OUT}/figures/CompareUQMethods_FIG_WhatMattersForTheChoice.png',
            bbox_inches='tight', dpi=300)
figstyle.check_overlaps(fig)
plt.show()

# THE SAME TEST ON THE CORPUS, as a supplement with its own message: the
# synthetic datasets cannot see these effects, because they do not span the
# shape variety the real categories do. That is lost POWER and not bias -- the
# corpus still covers 98 to 100 percent of real datasets on every
# characteristic -- and it is the coverage shortfall showing up where it costs
# something.
_gs = _ci[(_ci.base == 'n+coeffvar') & (_ci.arm == 'synthetic')]
fig, ax = plt.subplots(figsize=(figstyle.FULL_WIDTH * 0.78, 4.6))
for _i, _m in enumerate(_order):
    for _arm, _col, _mk, _src in (('empirical', figstyle.ACCENT, 'o', _g),
                                  ('synthetic', figstyle.MUTED, 'D', _gs)):
        _v = _src[_src.metric == _m].incremental_r2
        if len(_v):
            ax.scatter([_v.max()], [_i], s=28, marker=_mk, color=_col,
                       linewidths=0, zorder=3)
ax.set_yticks(_y)
ax.set_yticklabels([dct_metriclabels.get(m, m).replace('\n', ' ')
                    for m in _order], fontsize=6.5)
ax.set_xlabel('added explanatory power for which method fits better',
              fontsize=7.5)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(labelsize=6.5)
ax.set_xlim(left=0)
figstyle.label_line(ax, _g.incremental_r2.max(), len(_order) - 1,
                    'real ECC categories', color=figstyle.ACCENT, dx=7,
                    fontsize=6.5)
figstyle.label_line(ax, _gs.incremental_r2.max(), len(_order) - 2.2,
                    'synthetic corpus', color=figstyle.MUTED, dx=7,
                    fontsize=6.5)
ax.set_title('The corpus cannot see the effects the real data show, because '
             'it does not\nspan their shape variety', fontsize=9.5, loc='left')
plt.savefig(f'{OUT}/figures/CompareUQMethods_SUPP_ChoicePowerByArm.png',
            bbox_inches='tight', dpi=300)
figstyle.check_overlaps(fig)
plt.show()

In [ ]:
# FIGURE B: the direction and size of each surviving effect.
#
# THE MESSAGE: the kernel estimate wins on large datasets and loses on skewed,
# lognormal-looking ones, and the crossing point is where a practitioner's
# choice actually changes.
#
# One panel per surviving characteristic. The y axis is log(W1_KDE /
# W1_lognormal), so ZERO IS A TIE, below zero the kernel estimate is closer
# and above it the lognormal is -- which is the whole reading, and it needs no
# legend. Two lines per panel rather than six, because the ratio is taken
# within a weighting scheme. The band is a bootstrap over equal-count bins and
# the rug is the count over the bin width.
_WEIGHT_COLOR = {'Uniform': figstyle.MUTED, 'Variable': figstyle.ACCENT}


# THE PANELS COME FROM THE INCREMENTS, not from the permutation importance,
# so that this figure and Figure A rank the same quantity. Ranking them
# differently put the author's modality index second in one figure and absent
# from the other.
_inc_emp = red_choice_increments[
    (red_choice_increments.base == 'n+coeffvar')
    & (red_choice_increments.arm == 'empirical')]
DRIVERS = (_inc_emp.groupby('metric').incremental_r2.max()
           .sort_values(ascending=False).head(6).index.tolist())
print('panels, by incremental contribution:', ', '.join(DRIVERS))


def plot_choice_drivers(arm, value, fname, title, chars=None):
    ch = red_choice[arm]
    chars = chars or DRIVERS
    here = [c for c in chars if c in ch.columns
            and np.isfinite(pd.to_numeric(ch[c], errors='coerce')).sum() > 60]
    if not here:
        print(f'nothing to draw for {arm}')
        return
    ncols = min(len(here), 3)
    nrows = int(np.ceil(len(here) / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(figstyle.FULL_WIDTH, 2.6 * nrows),
                             gridspec_kw=dict(hspace=0.75, wspace=0.28),
                             squeeze=False)
    for k, metric in enumerate(here):
        ax = axes[k // ncols][k % ncols]
        how = metricred.METRIC_TRANSFORM.get(metric, 'identity')
        ax.axhline(0.0, color='black', linewidth=0.8, zorder=1)
        for wt in ('Uniform', 'Variable'):
            g = ch[ch.weighting == wt]
            x = metricred.transform(g[metric].to_numpy(float), how)
            y = g.log_ratio.to_numpy(float)
            b = metricred.binned_curve(x, y, n_bins=10, rng=rng)
            if len(b):
                ax.fill_between(b.x_center, b.lo, b.hi, step='mid',
                                color=_WEIGHT_COLOR[wt], alpha=0.18,
                                linewidth=0)
            l = metricred.lowess_curve(
                x, y, frac=0.5, rng=rng,
                bounds=(b.x_center.min(), b.x_center.max()) if len(b) else None)
            if len(l):
                ax.plot(l.x, l.fit, color=_WEIGHT_COLOR[wt], linewidth=1.4)
        if len(b):
            lo, hi = ax.get_ylim()
            w = (b.x_hi - b.x_lo).to_numpy(float)
            d = b['count'].to_numpy(float) / np.where(w > 0, w, np.nan)
            d = np.nan_to_num(d / np.nanmax(d))
            ax.bar(b.x_lo, height=0.08 * (hi - lo) * d, width=w, bottom=lo,
                   align='edge', color=figstyle.FAINT, linewidth=0, zorder=0)
            ax.set_ylim(lo, hi)
        # KEEP THE NEWLINE. A label like "Lognormal Fit (Shapiro-Francia)
        # (Var)" is 37 characters and three of them across one page collide;
        # the label file already carries a break for exactly this.
        ax.set_title(dct_metriclabels.get(metric, metric), fontsize=7.5)
        ax.set_xlabel(f'{metric}, {how} scale', fontsize=6.5)
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=6.5)
        if k % ncols == 0:
            # A y-label reads BOTTOM TO TOP, so the first words land at
            # the bottom of the axis, which is where the log ratio is
            # negative and the kernel estimate is the closer of the two.
            # Written the other way round this label contradicted the
            # title, which is a graphical-integrity failure and not a
            # wording preference.
            ax.set_ylabel('KDE closer  <->  lognormal closer',
                          fontsize=7)
    # direct labels on the first panel, where the two lines are furthest apart
    # THE WEIGHT KEY GOES BELOW THE PANELS. Placed at the end of a curve it
    # ran off the panel edge and was clipped mid-word, and it covered 6 to 7
    # percent of the plotted ink in its box -- both of which
    # `figstyle.check_overlaps` now reports.
    for _j, _wt in enumerate(('Uniform', 'Variable')):
        fig.text(0.015 + 0.18 * _j, -0.01, f'{_wt} weights',
                 color=_WEIGHT_COLOR[_wt], fontsize=7, ha='left', va='top')
    for k in range(len(here), nrows * ncols):
        axes[k // ncols][k % ncols].set_axis_off()
    fig.suptitle(title, fontsize=9.5, y=1.015, x=0.02, ha='left')
    plt.savefig(f'{OUT}/figures/{fname}', bbox_inches='tight', dpi=300)
    figstyle.check_overlaps(fig)
    plt.show()


plot_choice_drivers(
    'empirical', 'w1_cv',
    'CompareUQMethods_FIG_ChoiceDrivers.png',
    'Above the line the lognormal fits better, below it the kernel estimate '
    'does: real ECC categories')
plot_choice_drivers(
    'synthetic', 'w1_market',
    'CompareUQMethods_SUPP_ChoiceDrivers_Synthetic.png',
    'The same comparison on the corpus, against the known parent')

In [ ]:
# SUPPLEMENT: the rolling average against its replacement, kept so the two
# can be checked against each other.
#
# The left panel is the rolling mean over 250 datasets each side, exactly as
# the study drew it. The right is the same data with a bootstrap band from
# equal-count bins and a density rug. If the two tell different stories the
# difference is a finding, not an improvement.
_old_curves = pd.read_csv(f'{OUT}/tables/TABLE_MethodCurves.csv.gz')
_curves_disk = pd.read_csv(f'{OUT}/tables/TABLE_ReductionCurves.csv.gz')


def plot_old_vs_new(arm, metric, value, fname):
    how = metricred.METRIC_TRANSFORM.get(metric, 'identity')
    fig, axes = plt.subplots(1, 2, figsize=(figstyle.FULL_WIDTH, 2.7),
                             gridspec_kw=dict(wspace=0.26))
    o = _old_curves[(_old_curves.arm == arm) & (_old_curves.value == 'w1')
                    & (_old_curves.characteristic == metric)]
    for pewt in PEWT:
        p = o[o.method == pewt]
        if len(p):
            axes[0].plot(metricred.transform(p.x.to_numpy(float), how),
                         p.y_smooth, color=dct_colors[pewt], linewidth=1.2)
    axes[0].set_title('rolling mean, 250 datasets each side', fontsize=8)
    n = _curves_disk[(_curves_disk.arm == arm) & (_curves_disk.value == value)
                     & (_curves_disk.characteristic == metric)]
    for pewt in PEWT:
        pb = n[(n.kind == 'binned') & (n.method == pewt)].sort_values('x_center')
        if len(pb):
            axes[1].fill_between(pb.x_center, pb.lo, pb.hi, step='mid',
                                 color=dct_colors[pewt], alpha=0.16,
                                 linewidth=0)
        p = n[(n.kind == 'lowess') & (n.method == pewt)].sort_values('x_center')
        if len(p):
            axes[1].plot(p.x_center, p['mean'], color=dct_colors[pewt],
                         linewidth=1.2)
    axes[1].set_title('bootstrap band, and a rug where the datasets are',
                      fontsize=8)
    for ax in axes:
        ax.set_xlabel(f'{metric}, {how} scale', fontsize=7)
        ax.set_ylabel('W1', fontsize=7)
        ax.set_yscale('log')
        ax.spines[['top', 'right']].set_visible(False)
        ax.tick_params(labelsize=6.5)
    fig.suptitle(f'Same story, with the uncertainty shown: {metric}, '
                 f'{arm} arm', fontsize=9.5, y=1.04, x=0.02, ha='left')
    plt.savefig(f'{OUT}/figures/{fname}', bbox_inches='tight', dpi=300)
    plt.show()


plot_old_vs_new('empirical', 'n', 'w1_cv',
                'CompareUQMethods_SUPP_RollingVersusBinned_Empirical.png')
plot_old_vs_new('synthetic', 'n', 'w1_market',
                'CompareUQMethods_SUPP_RollingVersusBinned_Synthetic.png')